# Mejora 1: Calibración de threshold por clase

El threshold de 0.5 penaliza a las clases minoritarias (especialmente Infiltration).
Este notebook encuentra el threshold óptimo para cada clase usando el **val set**,
y los aplica al **test set** para reportar métricas mejoradas.

**Criterio de optimización:** Youden's J = Sensitivity + Specificity - 1.
Maximizarlo balancea la detección de positivos y negativos.

**Sin reentrenar.** Solo usa las probabilidades del checkpoint existente.
**Runtime: CPU. Tiempo: ~5 min.**


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR     = f'{PROJECT_ROOT}/code'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'

import os, sys, shutil, time
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

H5_LOCAL      = '/content/nih_images.h5'
PROCESSED_DIR = '/content/processed_s2'
CLASS_NAMES   = {0:'No Finding', 1:'Cardiomegaly', 2:'Effusion', 3:'Infiltration'}


In [ ]:
!pip install -q torchxrayvision scipy h5py
print('deps OK')


## 2. Verificar HDF5 y CSVs

In [ ]:
import h5py

if not (os.path.exists(H5_LOCAL) and os.path.getsize(H5_LOCAL) > 1_000_000_000):
    print('Copiando HDF5 desde Drive...')
    t0 = time.time()
    shutil.copy2(f'{NIH_DIR}/nih_images.h5', H5_LOCAL)
    print(f'✓ Copiado en {(time.time()-t0)/60:.1f} min')
else:
    with h5py.File(H5_LOCAL, 'r') as hf:
        print(f'✓ HDF5 local: {len(hf["images"]):,} imágenes')

import pandas as pd
os.makedirs(PROCESSED_DIR, exist_ok=True)
for csv in ['val.csv', 'test.csv']:
    local = f'{PROCESSED_DIR}/{csv}'
    if not os.path.exists(local):
        shutil.copy(f'{PROJECT_ROOT}/data/processed_s2/{csv}', local)
    print(f'✓ {csv}: {len(pd.read_csv(local))} filas')


## 3. Obtener probabilidades en val y test

In [ ]:
# Limpiar cache
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'): del sys.modules[mod]

import torch
import numpy as np
from torch.utils.data import DataLoader

from src.training.utils import load_config, get_device, load_checkpoint
from src.models.baseline import build_baseline_model
from src.datasets.nih_hdf5 import NIHDatasetHDF5
from src.datasets.transforms import build_eval_transform

cfg    = load_config(f'{CODE_DIR}/configs/sprint2.yaml')
device = get_device()

# Cargar modelo
model = build_baseline_model(cfg).to(device)
ckpt  = load_checkpoint(
    f'{PROJECT_ROOT}/experiments_s2/checkpoints/sprint2_best.pt',
    map_location=device
)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Modelo: epoch {ckpt["epoch"]} | val AUC_macro={ckpt["best_val_auc_macro"]:.4f}')

eval_tfm = build_eval_transform(224)

def get_probs_labels(csv_path):
    ds = NIHDatasetHDF5(csv_path, H5_LOCAL, eval_tfm)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0)
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch['image'].to(device))
            probs  = torch.softmax(logits.float(), dim=1)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(batch['label'].numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)

print('Obteniendo probabilidades en val...')
val_probs,  val_labels  = get_probs_labels(f'{PROCESSED_DIR}/val.csv')
print(f'  val:  {len(val_labels)} muestras')

print('Obteniendo probabilidades en test...')
test_probs, test_labels = get_probs_labels(f'{PROCESSED_DIR}/test.csv')
print(f'  test: {len(test_labels)} muestras')


## 4. Calibrar thresholds en val

Para cada clase, barremos thresholds de 0.05 a 0.70 y elegimos el que
maximiza el índice de Youden (Sensitivity + Specificity - 1).

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

optimal_thresholds = {}
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
colors = ['#2196F3','#F44336','#4CAF50','#FF9800']

print('CALIBRACIÓN DE THRESHOLDS EN VAL:')
print('=' * 65)

for i in range(4):
    y_bin = (val_labels == i).astype(int)
    fpr, tpr, thresholds = roc_curve(y_bin, val_probs[:, i])

    # Youden's J = TPR + TNR - 1 = Sensitivity + Specificity - 1
    tnr = 1 - fpr
    youden_j = tpr + tnr - 1
    best_idx  = np.argmax(youden_j)
    best_thr  = float(thresholds[best_idx])
    best_sens = float(tpr[best_idx])
    best_spec = float(tnr[best_idx])

    # Limitar threshold entre 0.05 y 0.70
    best_thr = max(0.05, min(0.70, best_thr))
    optimal_thresholds[i] = best_thr

    print(f'{CLASS_NAMES[i]:<15}: threshold={best_thr:.3f} | '
          f'Sens={best_sens:.4f} | Spec={best_spec:.4f} | '
          f'Youden={youden_j[best_idx]:.4f}')

    # Plot curva ROC + punto óptimo
    ax = axes[i]
    ax.plot(fpr, tpr, color=colors[i], lw=2)
    ax.scatter([fpr[best_idx]], [tpr[best_idx]],
               color='red', s=80, zorder=5,
               label=f'thr={best_thr:.2f}\nSens={best_sens:.2f}\nSpec={best_spec:.2f}')
    ax.plot([0,1],[0,1],'k--',alpha=0.3)
    ax.set_title(f'{CLASS_NAMES[i]}', color=colors[i], fontweight='bold')
    ax.set_xlabel('1-Spec (FPR)'); ax.set_ylabel('Sens (TPR)')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Calibración de thresholds (val) — Youden\'s J', fontsize=13)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/experiments_s2/figures/threshold_calibration.png',
            dpi=110, bbox_inches='tight')
plt.show()

print()
print(f'Thresholds óptimos: {optimal_thresholds}')


## 5. Aplicar thresholds calibrados al test set

In [ ]:
from src.evaluation.metrics_s2 import compute_metrics_multiclass

# Con threshold fijo 0.5 (baseline)
metrics_fixed = compute_metrics_multiclass(test_labels, test_probs)

# Con thresholds calibrados
# Para multi-clase: asignamos la clase cuya probabilidad supera su threshold
# y tiene la mayor probabilidad normalizada (en caso de empate o ninguno)
def predict_calibrated(probs, thresholds):
    n, c = probs.shape
    preds = np.zeros(n, dtype=int)
    for idx in range(n):
        # Probabilidades normalizadas por threshold
        scores = np.array([probs[idx, k] / thresholds[k] for k in range(c)])
        preds[idx] = np.argmax(scores)
    return preds

test_preds_cal = predict_calibrated(test_probs, optimal_thresholds)

# Crear una versión one-hot de probs para la función de métricas
# (la función acepta probs, no preds, para calcular AUC)
metrics_cal = compute_metrics_multiclass(test_labels, test_probs)
# Recalcular sens/spec/prec con los preds calibrados
from sklearn.metrics import confusion_matrix, accuracy_score
import numpy as np

def metrics_from_preds(y_true, y_pred, n_classes=4):
    acc = accuracy_score(y_true, y_pred)
    sens_list, spec_list, prec_list = [], [], []
    for c in range(n_classes):
        tp = int(((y_pred==c)&(y_true==c)).sum())
        fp = int(((y_pred==c)&(y_true!=c)).sum())
        tn = int(((y_pred!=c)&(y_true!=c)).sum())
        fn = int(((y_pred!=c)&(y_true==c)).sum())
        sens_list.append(tp/(tp+fn) if (tp+fn)>0 else 0.0)
        spec_list.append(tn/(tn+fp) if (tn+fp)>0 else 0.0)
        prec_list.append(tp/(tp+fp) if (tp+fp)>0 else 0.0)
    return acc, sens_list, spec_list, prec_list

# Preds con threshold fijo
test_preds_fixed = np.argmax(test_probs, axis=1)
acc_f, sens_f, spec_f, prec_f = metrics_from_preds(test_labels, test_preds_fixed)
acc_c, sens_c, spec_c, prec_c = metrics_from_preds(test_labels, test_preds_cal)

print('COMPARATIVA TEST: threshold fijo 0.5 vs calibrado')
print('=' * 70)
print(f"{'Métrica':<20} {'Fijo 0.5':>10} {'Calibrado':>10} {'Delta':>10}")
print('-' * 70)

print(f"{'Accuracy':<20} {acc_f:>10.4f} {acc_c:>10.4f} {acc_c-acc_f:>+10.4f}")
print(f"{'Sens macro':<20} {np.mean(sens_f):>10.4f} {np.mean(sens_c):>10.4f} {np.mean(sens_c)-np.mean(sens_f):>+10.4f}")
print(f"{'Spec macro':<20} {np.mean(spec_f):>10.4f} {np.mean(spec_c):>10.4f} {np.mean(spec_c)-np.mean(spec_f):>+10.4f}")
print()
print(f"{'Clase':<15} {'Sens 0.5':>9} {'Sens Cal':>9} {'Spec 0.5':>9} {'Spec Cal':>9}")
print('-' * 55)
for i in range(4):
    delta_s = sens_c[i]-sens_f[i]
    arrow   = '↑' if delta_s > 0.05 else ('↓' if delta_s < -0.05 else '≈')
    print(f"{CLASS_NAMES[i]:<15} {sens_f[i]:>9.4f} {sens_c[i]:>9.4f} "
          f"{spec_f[i]:>9.4f} {spec_c[i]:>9.4f}  {arrow} {delta_s:+.3f}")

print()
print(f'Nota: AUC no cambia (es independiente del threshold).')
print(f'AUC macro = {metrics_fixed["auc_macro"]:.4f} (igual con ambos métodos)')

# Guardar thresholds
import json
thr_path = f'{PROJECT_ROOT}/experiments_s2/optimal_thresholds.json'
with open(thr_path, 'w') as f:
    json.dump({'thresholds': optimal_thresholds,
               'criterion': 'youden_j',
               'calibrated_on': 'val'}, f, indent=2)
print(f'\n✓ Thresholds guardados: {thr_path}')


## 6. Matriz de confusión con thresholds calibrados

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, title in [
    (axes[0], test_preds_fixed,  'Threshold fijo 0.5'),
    (axes[1], test_preds_cal,    'Thresholds calibrados'),
]:
    cm = confusion_matrix(test_labels, preds, labels=list(range(4)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=[CLASS_NAMES[i] for i in range(4)],
                yticklabels=[CLASS_NAMES[i] for i in range(4)],
                cbar=False, square=True, ax=ax, annot_kws={'size':11})
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
    plt.setp(ax.get_xticklabels(), rotation=25, ha='right', fontsize=8)
    plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.suptitle('Comparativa: threshold fijo vs calibrado — Test Set', fontsize=13)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/experiments_s2/figures/cm_comparison.png',
            dpi=110, bbox_inches='tight')
plt.show()
print('✓ Guardado.')


## 7. Siguiente: Mejora 2 — Focal Loss

Con los thresholds calibrados ya aplicados, el siguiente paso es
**reentrenar con Focal Loss** para mejorar el AUC base de Infiltration y No Finding.

Vuelve al chat para recibir el notebook de Focal Loss.
